# Laboratório 8 — Spark: do DataFrame ao S3

Pipeline completo em PySpark: criar o DataFrame, controlar as partições, transformar com as
funções do dia a dia e gravar em Parquet particionado num S3 local (floci).

Rode as células **na ordem**. O notebook é idempotente: pode ser executado do início ao fim
quantas vezes quiser.

## 0. Ambiente

O Spark roda **na sua máquina**, no kernel do `.venv` desta pasta. Só o S3 (floci) está em Docker.

Esta célula resolve os três caminhos que o Windows precisa e falha cedo, com instrução, se
algum estiver faltando — em vez de deixar o erro aparecer lá na frente como um
`NoClassDefFoundError` sem contexto.

In [ ]:
Okay, thank you.import sys, os, platform
from pathlib import Path

BASE = Path.cwd()
if not (BASE / "jars").exists() and (BASE.parent / "jars").exists():
    BASE = BASE.parent

ENDPOINT = os.environ.get("FLOCI_ENDPOINT", "http://localhost:4567")
BUCKET   = "aula03-lake"

# 1. Java. O Spark nao sobe sem uma JVM 8, 11 ou 17.
try:
    import subprocess
    versao_java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
except FileNotFoundError:
    raise RuntimeError("Java nao encontrado no PATH. Instale um JDK 8, 11 ou 17.") from None

# 2. winutils.exe. So no Windows, e so quando os jars de S3 entram em cena:
#    a classe Shell do Hadoop procura esse binario na inicializacao estatica.
if platform.system() == "Windows":
    hadoop_home = BASE / "hadoop"
    if not (hadoop_home / "bin" / "winutils.exe").exists():
        raise RuntimeError(
            f"Falta {hadoop_home / 'bin' / 'winutils.exe'}.\n"
            "Rode o Passo 4 do README (baixa winutils.exe e hadoop.dll)."
        )
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["PATH"] = str(hadoop_home / "bin") + os.pathsep + os.environ["PATH"]

# 3. Os jars que ensinam o Hadoop a falar S3.
JARS = [BASE / "jars" / "hadoop-aws-3.3.4.jar",
        BASE / "jars" / "aws-java-sdk-bundle-1.12.262.jar"]
faltando = [j.name for j in JARS if not j.exists()]
if faltando:
    raise RuntimeError(f"Faltam jars em {BASE / 'jars'}: {faltando}\nRode o Passo 4 do README.")
JARS = ",".join(str(j) for j in JARS)

try:
    import pyspark
except ModuleNotFoundError:
    raise RuntimeError(
        f"pyspark nao esta instalado neste kernel.\nInterpretador: {sys.executable}\n"
        "Selecione o kernel .venv desta pasta (Passo 3 do README)."
    ) from None

# 4. O interpretador dos workers. Sem isto o Spark lanca o worker com o
#    `python` do PATH, que pode ser outra versao -- e o sintoma nao diz isso:
#    "Python worker failed to connect back".
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("python     :", sys.version.split()[0], "|", sys.executable)
print("pyspark    :", pyspark.__version__)
print("java       :", versao_java)
print("endpoint S3:", ENDPOINT)

## 1. A sessão Spark

Toda a configuração de S3 mora aqui. As variáveis `ENDPOINT` e `JARS` vêm da seção 0.

In [ ]:
import time, urllib.request, urllib.error, re, collections
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (SparkSession.builder
    .appName("aula03-spark")
    .master("local[4]")
    .config("spark.jars", JARS)
    .config("spark.hadoop.fs.s3a.endpoint", ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", "test")
    .config("spark.hadoop.fs.s3a.secret.key", "test")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.shuffle.partitions", "8")
    # No Windows o Spark nao usa daemon de workers: cada tarefa que precisa de
    # Python sobe um processo novo, e o padrao de 15 s para ele se conectar de
    # volta as vezes nao basta. O sintoma e "Python worker failed to connect back".
    .config("spark.python.authenticate.socketTimeout", "120")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
ms = lambda t: int((time.time() - t) * 1000)
print("Spark", spark.version, "| cores:", spark.sparkContext.defaultParallelism)

## 2. O bucket

O Spark grava objetos, mas não cria bucket. Um `PUT` na raiz resolve — e o `409` é tratado
para a célula poder rodar duas vezes.

In [ ]:
def criar_bucket(nome):
    req = urllib.request.Request(f"{ENDPOINT}/{nome}", method="PUT")
    try:
        with urllib.request.urlopen(req) as r:
            return f"criado (HTTP {r.status})"
    except urllib.error.HTTPError as e:
        if e.code == 409:
            return "ja existia (HTTP 409)"
        raise

print(criar_bucket(BUCKET))
print(criar_bucket(BUCKET))   # idempotente

## 3. Criar um DataFrame no código

`createDataFrame` com **schema explícito**. Declarar o schema evita que o Spark leia os dados
duas vezes para adivinhar os tipos — e evita que ele adivinhe errado.

In [ ]:
schema = StructType([
    StructField("pedido_id", IntegerType(), False),
    StructField("cliente",   StringType(),  False),
    StructField("categoria", StringType(),  False),
    StructField("valor",     DoubleType(),  False),
])

linhas = [(1, "ana",   "eletronicos", 1200.0),
          (2, "bruno", "livros",        89.9),
          (3, "ana",   "casa",         340.5),
          (4, "carla", "eletronicos", 2100.0)]

pequeno = spark.createDataFrame(linhas, schema)
pequeno.show()
pequeno.printSchema()

## 4. Um DataFrame com volume

Quatro linhas não mostram nada sobre partição. `spark.range` gera 2 milhões de linhas já
distribuídas, e as colunas saem de funções nativas — sem `for`, sem `append`.

In [ ]:
CATS = ["eletronicos", "papelaria", "casa", "vestuario", "livros"]
UFS  = ["SP", "RJ", "MG", "BA", "PE", "RS"]

def sorteia(valores, semente):
    return F.element_at(F.array(*[F.lit(v) for v in valores]),
                        (F.rand(semente) * len(valores) + 1).cast("int"))

pedidos = (spark.range(0, 2_000_000).withColumnRenamed("id", "pedido_id")
    .withColumn("cliente_id", (F.rand(7) * 50000).cast("int"))
    .withColumn("categoria",  sorteia(CATS, 1))
    .withColumn("quantidade", (F.rand(2) * 9 + 1).cast("int"))
    .withColumn("valor",      F.round(F.rand(3) * 990 + 10, 2))
    .withColumn("uf",         sorteia(UFS, 4))
    .withColumn("data",       F.expr("date_add(to_date('2026-01-01'), cast(rand(5) * 364 as int))"))
    .withColumn("status",     F.when(F.rand(6) < 0.08, "cancelado").otherwise("pago")))

pedidos.cache()
print("linhas:", pedidos.count())
pedidos.show(5)

## 5. Partições

A partição é a unidade de paralelismo: uma partição, uma tarefa, um core por vez.

`glom()` devolve o conteúdo real de cada partição — diferente de `getNumPartitions()`, ele
não pode ser reescrito pelo otimizador.

In [ ]:
def perfil(rotulo, df):
    tamanhos = df.rdd.glom().map(len).collect()
    print(f"{rotulo:<24} {len(tamanhos):>3} particoes  {tamanhos}")

perfil("original",            pedidos)
perfil("repartition(12)",     pedidos.repartition(12))
perfil("coalesce(2)",         pedidos.coalesce(2))
perfil("repartition('uf')",   pedidos.repartition("uf"))
perfil("repartition(3,'uf')", pedidos.repartition(3, "uf"))

Repare nas duas últimas linhas: particionar **por coluna** não dá partições iguais. São 6 UFs
distribuídas por hash em 8 espaços — duas caem no mesmo, e a partição resultante fica com o
dobro das outras. Esse é o desequilíbrio (*skew*) que faz uma tarefa demorar o triplo das demais.

## 6. As funções de transformação

### `select` · `filter` · `withColumn` · `when`

In [ ]:
enriquecido = (pedidos
    .filter(F.col("status") == "pago")
    .withColumn("total", F.round(F.col("valor") * F.col("quantidade"), 2))
    .withColumn("faixa", F.when(F.col("total") >= 4000, "A")
                          .when(F.col("total") >= 1500, "B")
                          .otherwise("C"))
    .withColumn("mes", F.date_format("data", "yyyy-MM"))
    .select("pedido_id", "cliente_id", "categoria", "uf", "mes",
            "quantidade", "valor", "total", "faixa"))

enriquecido.show(5)

### `groupBy` · `agg` · `orderBy`

O `cast("decimal(18,2)")` existe para o número sair legível: soma de `double` grande é
impressa em notação científica.

In [ ]:
resumo = (enriquecido.groupBy("uf", "categoria")
    .agg(F.count("*").alias("pedidos"),
         F.round(F.sum("total"), 2).cast("decimal(18,2)").alias("receita"),
         F.round(F.avg("total"), 2).alias("ticket"),
         F.max("total").alias("maior"))
    .orderBy(F.col("receita").desc()))

resumo.show(5)

### `join`

`broadcast` manda a tabela pequena inteira para cada executor e elimina o embaralhamento do
lado grande. Vale enquanto a tabela couber na memória do executor.

In [ ]:
clientes = (spark.range(0, 50_000).withColumnRenamed("id", "cliente_id")
    .withColumn("cliente_id", F.col("cliente_id").cast("int"))
    .withColumn("segmento", sorteia(["varejo", "atacado", "corporativo"], 8)))

com_segmento = enriquecido.join(F.broadcast(clientes), "cliente_id", "inner")

(com_segmento.groupBy("segmento")
    .agg(F.count("*").alias("pedidos"),
         F.round(F.sum("total"), 2).cast("decimal(18,2)").alias("receita"))
    .orderBy("segmento").show())

### Funções de janela

Agregam **sem colapsar as linhas**: cada linha continua existindo e ganha o resultado do
grupo ao lado.

In [ ]:
janela = Window.partitionBy("uf").orderBy(F.col("receita").desc())

top2 = (resumo
    .withColumn("posicao", F.row_number().over(janela))
    .withColumn("dif_para_1o", F.round(F.col("receita") - F.first("receita").over(janela), 2))
    .filter(F.col("posicao") <= 2))

top2.orderBy("uf", "posicao").show(6)

## 7. UDF: onde o otimizador para de te ajudar

Uma UDF em Python é uma caixa-preta para o Catalyst. Ele não consegue reescrevê-la, e cada
linha precisa cruzar a fronteira JVM ↔ Python.

A primeira medição abaixo está **errada** de propósito.

In [ ]:
def medir(rotulo, fn, rodadas=3):
    tempos = []
    for _ in range(rodadas):
        t = time.time(); fn(); tempos.append(ms(t))
    tempos.sort()
    print(f"{rotulo:<32} {tempos}  mediana={tempos[1]} ms")
    return tempos[1]

udf_categoria = F.udf(lambda c: c.upper() if c else None, StringType())

print("-- ERRADO: com .count() o otimizador apaga a projecao inteira")
medir("nativa + count()", lambda: enriquecido.select(F.upper("categoria").alias("c")).count())
medir("UDF    + count()", lambda: enriquecido.select(udf_categoria("categoria").alias("c")).count())

print()
print("-- CERTO: agregar a coluna obriga o calculo a acontecer")
a = medir("nativa: F.upper()",     lambda: enriquecido.select(F.upper("categoria").alias("c")).agg(F.max("c")).collect())
b = medir("UDF Python: c.upper()", lambda: enriquecido.select(udf_categoria("categoria").alias("c")).agg(F.max("c")).collect())
print(f"\n--> a UDF custou {b/a:.1f}x a versao nativa")

## 8. Gravar no S3

`partitionBy` cria a estrutura de pastas `uf=SP/`, `uf=RJ/`… — é o que permite ao leitor
pular arquivos inteiros depois.

In [ ]:
saida = enriquecido.select("pedido_id", "categoria", "uf", "mes",
                           "quantidade", "valor", "total")

t = time.time()
(saida.write.mode("overwrite")
      .partitionBy("uf")
      .parquet(f"s3a://{BUCKET}/curated/pedidos"))
print("write partitionBy('uf'):", ms(t), "ms")

### Quantos arquivos isso gerou?

In [ ]:
def listar(prefixo):
    url = f"{ENDPOINT}/{BUCKET}?list-type=2&prefix={prefixo}&max-keys=1000"
    with urllib.request.urlopen(url) as r:
        chaves = re.findall(r"<Key>([^<]+)</Key>", r.read().decode())
    parquet = [k for k in chaves if k.endswith(".parquet")]
    nivel = len(prefixo.rstrip("/").split("/"))
    return len(parquet), dict(sorted(collections.Counter(k.split("/")[nivel] for k in parquet).items()))

n, por_pasta = listar("curated/pedidos/")
print(f"{n} arquivos parquet  {por_pasta}")

# a mesma escrita, agrupando antes por particao
(saida.repartition("uf").write.mode("overwrite")
      .partitionBy("uf")
      .parquet(f"s3a://{BUCKET}/curated/pedidos_agrupado"))
n2, por_pasta2 = listar("curated/pedidos_agrupado/")
print(f"{n2} arquivos parquet  {por_pasta2}")

Quatro partições em memória × seis pastas = 24 arquivos. Com `repartition("uf")` antes da
escrita, cada UF vira uma partição só e sai **um** arquivo por pasta.

Muitos arquivos pequenos custam caro em object storage: cada um é uma requisição HTTP na leitura.

## 9. Ler de volta

O filtro por `uf` não lê os arquivos das outras UFs — o caminho já diz o que tem dentro
(*partition pruning*).

In [ ]:
t = time.time()
volta = spark.read.parquet(f"s3a://{BUCKET}/curated/pedidos")
total = volta.count()
print("leitura completa .............", ms(t), "ms ->", total, "linhas")

t = time.time()
sp = spark.read.parquet(f"s3a://{BUCKET}/curated/pedidos").filter(F.col("uf") == "SP").count()
print("leitura com filtro uf='SP' ...", ms(t), "ms ->", sp, "linhas")

print()
print("colunas na volta:", volta.columns)

Repare que `uf` aparece **por último**. Ela não está gravada dentro dos arquivos: o Spark a
reconstrói a partir do nome da pasta. A coluna de particionamento vira metadado do caminho.

In [ ]:
spark.stop()
print("sessao encerrada")